# Synthetic Decision Boundaries

Synthetic datasets are the cleanest way to study decision boundary geometry
because the ground-truth structure is known in advance.  This notebook embeds
classical 2-D classification patterns (moons, concentric circles, interleaved
blobs) into a 20-dimensional space by appending pure noise features, then uses
sensitivity projection to recover the two informative axes from the noise.

The fact that the boundaries are recovered cleanly validates that the sensitivity
projection is finding structure the model learned — not structure in the data.

In [ ]:
!pip install -q geolatent

In [ ]:
import numpy as np
import plotly.io as pio
from sklearn.datasets import make_moons, make_circles, make_blobs
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from geolatent import visualize_decision_geometry

pio.renderers.default = "colab"
rng = np.random.default_rng(0)

def embed(X_2d, D=20, noise_scale=0.3):
    """Embed 2-D signal into D dimensions by appending noise features."""
    noise = rng.normal(scale=noise_scale, size=(len(X_2d), D - 2))
    return np.hstack([X_2d, noise])

---
## Two Moons — MLP

Two interleaved half-circles in 20-D.  A wide MLP learns the non-linear
separation; the sensitivity axes should collapse onto the two informative
dimensions, recovering the crescent boundary cleanly.

In [ ]:
X_moons, y_moons = make_moons(n_samples=600, noise=0.15, random_state=0)
X_moons_20 = embed(X_moons)

mlp_moons = Pipeline([
    ("scaler", StandardScaler()),
    ("mlp", MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=400, random_state=0)),
]).fit(X_moons_20, y_moons)

feature_names = ["x", "y"] + [f"noise_{i}" for i in range(18)]

visualize_decision_geometry(
    mlp_moons, X_moons_20, y_moons,
    projection_method="sensitivity",
    feature_names=feature_names,
    class_names={0: "Moon A", 1: "Moon B"},
    show_confidence=True,
    show_centroids=True,
    show_ellipsoids=True,
    title="Two Moons (20-D) — MLP sensitivity projection",
).show()

---
## Concentric Circles — RBF SVM

A radially symmetric problem that linear models cannot solve.  The RBF kernel
implicitly lifts the data to infinite dimensions; the sensitivity projection
shows which directions in input space carry the ring-like boundary structure.

In [ ]:
X_circles, y_circles = make_circles(n_samples=600, noise=0.08, factor=0.4, random_state=0)
X_circles_20 = embed(X_circles)

svm_circles = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf", C=5.0, gamma="scale", probability=True, random_state=0)),
]).fit(X_circles_20, y_circles)

visualize_decision_geometry(
    svm_circles, X_circles_20, y_circles,
    projection_method="sensitivity",
    feature_names=feature_names,
    class_names={0: "Inner", 1: "Outer"},
    show_confidence=True,
    show_centroids=True,
    show_ellipsoids=True,
    title="Concentric Circles (20-D) — RBF SVM sensitivity projection",
).show()

---
## Four Anisotropic Blobs — GBM

Blobs with different spreads and orientations.  The cluster covariances are
anisotropic, meaning the boundaries are non-equidistant between centroids.
Mahalanobis ellipsoids show the true covariance shape of each class.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

centers = [[-3, -3], [-3, 3], [3, -3], [3, 3]]
cluster_std = [0.5, 1.5, 0.8, 2.0]
X_blobs, y_blobs = make_blobs(n_samples=800, centers=centers, cluster_std=cluster_std, random_state=0)
X_blobs_20 = embed(X_blobs, noise_scale=0.5)

gbm_blobs = Pipeline([
    ("scaler", StandardScaler()),
    ("gbm", GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=0)),
]).fit(X_blobs_20, y_blobs)

visualize_decision_geometry(
    gbm_blobs, X_blobs_20, y_blobs,
    projection_method="sensitivity",
    feature_names=feature_names,
    class_names={0: "Tight-A", 1: "Wide-B", 2: "Mid-C", 3: "Diffuse-D"},
    show_confidence=True,
    show_centroids=True,
    show_ellipsoids=True,
    title="Anisotropic Blobs (20-D) — GBM sensitivity projection",
).show()

---
## XOR Pattern — MLP

The XOR problem requires at least one hidden layer to solve.  Four clusters
are placed at the quadrant corners with alternating labels, creating a
checkerboard boundary that no linear model can represent.

In [ ]:
xor_centers = [[-2, -2], [-2, 2], [2, -2], [2, 2]]
X_xor_2d, _ = make_blobs(n_samples=800, centers=xor_centers, cluster_std=0.6, random_state=1)
y_xor = np.array([0, 1, 1, 0]).repeat(200)  # XOR labeling
X_xor_20 = embed(X_xor_2d, noise_scale=0.4)

mlp_xor = Pipeline([
    ("scaler", StandardScaler()),
    ("mlp", MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=0)),
]).fit(X_xor_20, y_xor)

visualize_decision_geometry(
    mlp_xor, X_xor_20, y_xor,
    projection_method="sensitivity",
    feature_names=feature_names,
    class_names={0: "Class 0", 1: "Class 1"},
    show_confidence=True,
    show_centroids=True,
    show_ellipsoids=True,
    title="XOR Pattern (20-D) — MLP sensitivity projection",
).show()